# Qwen



In [ ]:
# =========================
# 1. 安装依赖
# =========================
!pip -q install dashscope pymupdf pillow matplotlib

# =========================
# 2. 导入库
# =========================
import os
import fitz  # PyMuPDF
import dashscope
import matplotlib.pyplot as plt

from google.colab import files
from dashscope import MultiModalConversation

# =========================
# 3. 配置 API Key
# =========================
# 方式1：直接输入（推荐在 Colab 临时使用）
api_key = input("请输入你的阿里云百炼 DASHSCOPE_API_KEY: ").strip()
os.environ["DASHSCOPE_API_KEY"] = api_key

# 阿里云文档中的中国内地默认 API 地址
dashscope.base_http_api_url = "https://dashscope.aliyuncs.com/api/v1"

# =========================
# 4. 上传本地 PDF
# =========================
print("请上传一个 PDF 文件")
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("已上传文件：", pdf_path)

# =========================
# 5. 将 PDF 第一页渲染为 PNG
# =========================
doc = fitz.open(pdf_path)
page = doc.load_page(0)  # 第 1 页，索引为 0

# 提高清晰度：2倍缩放
matrix = fitz.Matrix(2, 2)
pix = page.get_pixmap(matrix=matrix, alpha=False)

first_page_img = "pdf_first_page.png"
pix.save(first_page_img)
doc.close()

print("第一页已保存为：", first_page_img)

# =========================
# 6. 显示第一页图片
# =========================
img = plt.imread(first_page_img)
plt.figure(figsize=(10, 14))
plt.imshow(img)
plt.axis("off")
plt.title("PDF 第1页渲染结果")
plt.show()

# =========================
# 7. 构造本地文件路径
# Colab 是 Linux 环境，阿里云文档说明 Python SDK 本地文件应使用:
# file://{绝对路径}
# =========================
abs_img_path = os.path.abspath(first_page_img)
image_path = f"file://{abs_img_path}"

print("传给模型的本地图片路径：", image_path)

# =========================
# 8. 调用 Qwen-VL OCR 解析第一页
# =========================
messages = [
    {
        "role": "user",
        "content": [
            {"image": image_path},
            {
                "text": (
                    "请解析这张PDF第一页图片的内容。"
                    "要求：\n"
                    "1. 尽可能完整提取页面中的文字；\n"
                    "2. 保持原有结构；\n"
                    "3. 如果有标题、表格、编号、项目符号，请按层次整理；\n"
                    "4. 如果部分内容看不清，请明确标注。"
                )
            }
        ]
    }
]

response = MultiModalConversation.call(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model="qwen-vl-ocr",
    messages=messages
)

# =========================
# 9. 解析返回结果
# =========================
def extract_text_from_response(resp):
    """
    尽量兼容不同返回格式
    """
    try:
        content = resp.output.choices[0].message.content
        if isinstance(content, str):
            return content

        if isinstance(content, list):
            texts = []
            for item in content:
                if isinstance(item, dict):
                    if "text" in item:
                        texts.append(item["text"])
                    else:
                        texts.append(str(item))
                else:
                    texts.append(str(item))
            return "\n".join(texts)

        return str(content)
    except Exception:
        return str(resp)

result_text = extract_text_from_response(response)

print("\n" + "=" * 80)
print("Qwen 解析结果：")
print("=" * 80)
print(result_text)

# =========================
# 10. 如需保存结果到 txt
# =========================
with open("pdf_first_page_parse.txt", "w", encoding="utf-8") as f:
    f.write(result_text)

print("\n解析结果已保存到: pdf_first_page_parse.txt")